In [12]:
%pip install pytest

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 2
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path
import reader
import validator
import transformer
print("Modules imported successfully!")

Modules imported successfully!


In [4]:
INPUT_FOLDER = Path(
    r"C:\ProgramData\MySQL\MySQL Server 8.0\Uploads"
)
CONFIG_PATH = Path("config.json")
ORDERS_FILE = INPUT_FOLDER / "olist_orders_dataset.csv"
ORDER_ITEMS_FILE = INPUT_FOLDER / "olist_order_items_dataset.csv"

In [ ]:
# 3
orders = reader.read_csv(ORDERS_FILE)
order_items = reader.read_csv(ORDER_ITEMS_FILE)
print("Orders records:", len(orders))
print("Order items records:", len(order_items))
print(orders[:2])

Orders records: 99441
Order items records: 112650
[{'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'customer_id': '9ef432eb6251297304e76186b10a928d', 'order_status': 'delivered', 'order_purchase_timestamp': '2017-10-02 10:56:33', 'order_approved_at': '2017-10-02 11:07:15', 'order_delivered_carrier_date': '2017-10-04 19:55:00', 'order_delivered_customer_date': '2017-10-10 21:25:13', 'order_estimated_delivery_date': '2017-10-18 00:00:00'}, {'order_id': '53cdb2fc8bc7dce0b6741e2150273451', 'customer_id': 'b0830fb4747a6c6d20dea0b8c802d7ef', 'order_status': 'delivered', 'order_purchase_timestamp': '2018-07-24 20:41:37', 'order_approved_at': '2018-07-26 03:24:27', 'order_delivered_carrier_date': '2018-07-26 14:31:00', 'order_delivered_customer_date': '2018-08-07 15:27:45', 'order_estimated_delivery_date': '2018-08-13 00:00:00'}]


In [6]:
# 4
orders = validator.normalize_columns(orders)
order_items = validator.normalize_columns(order_items)
print("Orders columns:")
print(orders[0].keys())
print("\nOrder items columns:")
print(order_items[0].keys())

Orders columns:
dict_keys(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'])

Order items columns:
dict_keys(['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value'])


In [8]:
# 5
orders = transformer.standardize_columns(
    orders,
    config["column_mapping"]["orders"]
)
order_items = transformer.standardize_columns(
    order_items,
    config["column_mapping"]["order_items"]
)
print("Standardized orders:")
print(orders[:2])
print("\nStandardized order items:")
print(order_items[:2])

Standardized orders:
[{'order_id': 'e481f51cbdc54678b7cc49136f2d6af7', 'customer_id': '9ef432eb6251297304e76186b10a928d', 'status': 'delivered', 'purchase_timestamp': '2017-10-02 10:56:33', 'approved_timestamp': '2017-10-02 11:07:15', 'carrier_date': '2017-10-04 19:55:00', 'delivered_timestamp': '2017-10-10 21:25:13', 'estimated_delivery_timestamp': '2017-10-18 00:00:00'}, {'order_id': '53cdb2fc8bc7dce0b6741e2150273451', 'customer_id': 'b0830fb4747a6c6d20dea0b8c802d7ef', 'status': 'delivered', 'purchase_timestamp': '2018-07-24 20:41:37', 'approved_timestamp': '2018-07-26 03:24:27', 'carrier_date': '2018-07-26 14:31:00', 'delivered_timestamp': '2018-08-07 15:27:45', 'estimated_delivery_timestamp': '2018-08-13 00:00:00'}]

Standardized order items:
[{'order_id': '00010242fe8c5a6d1ba2dd792cb16214', 'item_id': '1', 'product_id': '4244733e06e7ecb4970a6e2683c13e61', 'seller_id': '48436dade18ac8b2bce089ec2a041202', 'shipping_limit_date': '2017-09-19 09:45:35', 'price': '58.90', 'freight': '13

In [9]:
# 6
orders = transformer.convert_data_types(
    orders,
    config["data_types"]
)
order_items = transformer.convert_data_types(
    order_items,
    config["data_types"]
)
print("Price type:", type(order_items[0]["price"]))
print("Freight type:", type(order_items[0]["freight"]))
print(
    "Timestamp type:",
    type(orders[0]["purchase_timestamp"])
)

Price type: <class 'float'>
Freight type: <class 'float'>
Timestamp type: <class 'datetime.datetime'>


In [10]:
# 7
order_totals = transformer.calculate_order_totals(
    order_items
)
print("Total number of orders:", len(order_totals))
print("\nFirst 10 order totals:")
for order_id, total in list(order_totals.items())[:10]:
    print(order_id, ":", total)

Total number of orders: 98666

First 10 order totals:
00010242fe8c5a6d1ba2dd792cb16214 : 72.19
00018f77f2f0320c557190d7a144bdd3 : 259.83
000229ec398224ef6ca0657da4fc703e : 216.87
00024acbcdf0a6daa1e931b038114c75 : 25.78
00042b26cf59d7ce69dfabb4e55b4fd9 : 218.04000000000002
00048cc3ae777c65dbb7d2a0634bc1ea : 34.589999999999996
00054e8431b9d7675808bcb819fb4a32 : 31.75
000576fe39319847cbb9d288c5617fa6 : 880.75
0005a1a1728c9d785b8e2b08b904576c : 157.6
0005f50442cb953dcd1d21e1fb923495 : 65.39


In [16]:
# 8
import sys
import subprocess
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "tests/test_transformer.py"],
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

============================= test session starts =============================
platform win32 -- Python 3.14.6, pytest-9.0.3, pluggy-1.6.0 -- c:\Users\Viveka Shree\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\Viveka Shree\Desktop\Python Training
plugins: anyio-4.12.1
collecting ... collected 3 items

tests/test_transformer.py::test_to_snake_case PASSED                     [ 33%]
tests/test_transformer.py::test_standardize_columns PASSED               [ 66%]
tests/test_transformer.py::test_calculate_order_totals PASSED            [100%]

============================== 3 passed in 0.04s ==============================




In [7]:
# 10
config = transformer.load_config(CONFIG_PATH)
print(config)

{'column_mapping': {'orders': {'order_id': 'order_id', 'customer_id': 'customer_id', 'order_status': 'status', 'order_purchase_timestamp': 'purchase_timestamp', 'order_approved_at': 'approved_timestamp', 'order_delivered_carrier_date': 'carrier_date', 'order_delivered_customer_date': 'delivered_timestamp', 'order_estimated_delivery_date': 'estimated_delivery_timestamp'}, 'order_items': {'order_id': 'order_id', 'order_item_id': 'item_id', 'product_id': 'product_id', 'seller_id': 'seller_id', 'price': 'price', 'freight_value': 'freight'}}, 'data_types': {'price': 'float', 'freight': 'float', 'purchase_timestamp': 'datetime', 'approved_timestamp': 'datetime', 'carrier_date': 'datetime', 'delivered_timestamp': 'datetime', 'estimated_delivery_timestamp': 'datetime'}}


In [ ]:
#9
%load_ext autoreload
%autoreload 2

import reader
import validator
import transformer

print("All modules imported successfully!")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
All modules imported successfully!


In [18]:
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "tests/test_transformer.py"]
)

CompletedProcess(args=['c:\\Users\\Viveka Shree\\anaconda3\\python.exe', '-m', 'pytest', '-v', 'tests/test_transformer.py'], returncode=0)